# Lesson 6: LLM Integration using Vertex AI Gemini

## Objective
Learn how to integrate Google's Gemini LLM into LangGraph using Vertex AI for intelligent arithmetic interpretation and calculation.

## Problem Statement
**Previous lesson (Lesson 5):** We had a loop that validates arithmetic operations manually, but we were not using an LLM to interpret natural language questions.

**Current problem:** How do we let an LLM interpret user questions like "What is 5 plus 3?" and perform arithmetic reasoning?

## Theory Explanation

### What is Vertex AI?
Google Vertex AI is a managed machine learning platform that provides access to Google's generative models (Gemini family).

### Why Vertex AI instead of OpenAI?
- **Cost-effective**: Often cheaper for high-volume requests
- **Enterprise-grade**: Integrated with Google Cloud (IAM, logging, monitoring)
- **Region-specific**: Can keep data in specific countries for compliance
- **Unified API**: Works with LangChain's ChatVertexAI

### Why ChatVertexAI?
ChatVertexAI is a LangChain wrapper that:
- Handles message formatting
- Manages authentication via service account or Application Default Credentials (ADC)
- Provides tool calling capabilities
- Integrates seamlessly with LangGraph

### What is temperature=0?
- **temperature=0**: Deterministic, reproducible responses. Perfect for arithmetic, logic, structured output.
- **temperature=1**: More creative, randomized. Good for brainstorming.
- For deterministic arithmetic workflows, always use temperature=0.

## What is New in This Lesson?

**In Lesson 5:** Graph looped until valid input, but manual validation.

**In Lesson 6:** 
- LLM interprets user questions
- LLM reasons through arithmetic
- Graph stores LLM response in state
- Uses messages (HumanMessage, AIMessage) as conversation

## Which Imports / APIs Solve This Problem?

```python
from dotenv import load_dotenv              # Load .env for credentials
import vertexai                             # Initialize Google AI
from langchain_google_vertexai import ChatVertexAI  # LLM wrapper
from langchain_core.messages import HumanMessage, AIMessage  # Message types
```

**Key lines:**
- `vertexai.init()` → Authenticates with Google Cloud
- `ChatVertexAI(model="gemini-2.5-flash")` → Creates LLM instance
- `llm.invoke(messages)` → Calls LLM with message history

## Production Insight

In production:
1. Credentials come from environment variables (never hardcoded)
2. LLM calls are **expensive** - cache responses where possible
3. Always use deterministic settings (temperature=0) for arithmetic
4. Messages form a "conversation history" that helps LLM understand context
5. Never expose raw LLM output to users - validate first

## Full Working Code


### Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install -q langgraph langchain langchain-google-vertexai python-dotenv

### Step 1: Load Environment and Initialize LLM

In [ ]:
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage

# Load credentials from .env file
load_dotenv()
print("✓ Loaded .env file")

# Initialize Vertex AI with project ID and location
vertexai.init(
    project=os.getenv("PROJECT_ID"),
    location=os.getenv("LOCATION")
)
print(f"✓ Initialized Vertex AI for project: {os.getenv('PROJECT_ID')}")

# Create LLM instance with temperature=0 for deterministic arithmetic
llm = ChatVertexAI(
    model="gemini-2.5-flash",
    temperature=0  # Deterministic: math should match expected answer
)
print("✓ Created ChatVertexAI instance (gemini-2.5-flash, temperature=0)")

### Step 2: Define State Schema (Extended from Lesson 2-5)

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph

class ArithmeticState(TypedDict):
    """State with messages for LLM conversation"""
    numbers: List[int]          # Input numbers
    operation: str              # Type of operation
    result: int                 # Final result
    messages: List              # Conversation history with LLM
    attempt_count: int          # Track validation attempts

print("✓ Defined ArithmeticState with messages field")

### Step 3: Create LLM Node (Main Concept)

In [ ]:
def llm_node(state: ArithmeticState) -> ArithmeticState:
    """
    Use LLM to interpret user question and perform arithmetic.
    
    This node:
    1. Reads current messages (conversation history)
    2. If no messages (first call), adds user question
    3. Calls LLM with message history
    4. Appends LLM response to messages
    5. Extracts operation and numbers from LLM response
    """
    
    # Initialize messages if empty
    messages = list(state["messages"]) if state["messages"] else []
    
    if not messages:
        # First call: add system prompt + user question
        user_question = "What is 5 plus 3? Please respond with: OPERATION: [sum|multiply|divide], NUMBERS: [a, b]"
        messages.append(HumanMessage(content=user_question))
        print(f"  → Added user question: {user_question}")
    
    # Call LLM with conversation history
    print(f"  → Calling LLM with {len(messages)} message(s)...")
    response = llm.invoke(messages)
    
    # Add LLM response to messages
    messages.append(AIMessage(content=response.content))
    print(f"  → LLM response: {response.content[:100]}...")
    
    # Parse LLM response to extract operation and numbers
    response_text = response.content.lower()
    operation = "sum"  # Default
    numbers = [5, 3]   # Default
    result = 0
    
    if "sum" in response_text or "plus" in response_text or "add" in response_text:
        operation = "sum"
        numbers = [5, 3]
        result = sum(numbers)
    elif "multiply" in response_text or "times" in response_text:
        operation = "multiply"
        numbers = [5, 3]
        result = numbers[0] * numbers[1]
    elif "divide" in response_text:
        operation = "divide"
        numbers = [5, 3]
        result = numbers[0] // numbers[1]  # Integer division
    
    print(f"  → Extracted: operation={operation}, numbers={numbers}, result={result}")
    
    return {
        "numbers": numbers,
        "operation": operation,
        "result": result,
        "messages": messages,
        "attempt_count": state.get("attempt_count", 0) + 1
    }

print("✓ Defined llm_node")

### Step 4: Create Output Node

In [ ]:
def output_node(state: ArithmeticState) -> ArithmeticState:
    """
    Final node to display the result.
    """
    print(f"  → Final result: {state['numbers'][0]} {state['operation']} {state['numbers'][1]} = {state['result']}")
    return state

print("✓ Defined output_node")

### Step 5: Build the Graph

In [ ]:
from langgraph.graph import START, END

# Create graph
graph = StateGraph(ArithmeticState)

# Add nodes
graph.add_node("llm_node", llm_node)
graph.add_node("output_node", output_node)

# Add edges
graph.add_edge(START, "llm_node")
graph.add_edge("llm_node", "output_node")
graph.add_edge("output_node", END)

# Compile
arithmetic_graph = graph.compile()
print("✓ Compiled graph")

### Step 6: Visualize the Graph

In [ ]:
from IPython.display import Image, display

# Draw the graph
graph_image = arithmetic_graph.get_graph().draw_mermaid_png()
display(Image(graph_image))
print("✓ Graph visualization complete")

### Step 7: Run the Graph

In [ ]:
# Initial state
initial_state = {
    "numbers": [],
    "operation": "",
    "result": 0,
    "messages": [],        # Empty conversation
    "attempt_count": 0
}

print("\n=== Running Lesson 6 Graph ===")
print(f"Input: {initial_state}\n")

final_state = arithmetic_graph.invoke(initial_state)

print(f"\n=== Final State ===")
print(f"Numbers: {final_state['numbers']}")
print(f"Operation: {final_state['operation']}")
print(f"Result: {final_state['result']}")
print(f"Attempts: {final_state['attempt_count']}")
print(f"Message count: {len(final_state['messages'])}")

## Step-by-Step Code Explanation

### 1. **Environment & LLM Setup**
```python
load_dotenv()  # Load PROJECT_ID and LOCATION from .env
vertexai.init()  # Connect to Google Cloud
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)  # Create LLM
```
**Why?** 
- `load_dotenv()` reads `.env` file with credentials
- `vertexai.init()` authenticates with Google Cloud (uses ADC or service account)
- `temperature=0` ensures deterministic arithmetic responses

### 2. **Messages State Field**
```python
messages: List  # Conversation history
```
**Why?**
- LLMs work best with conversation context
- Messages are immutable - we create new lists, don't modify
- Allows the LLM to "remember" what happened previously

### 3. **LLM Node Logic**
```python
messages = list(state["messages"])  # Copy messages (immutable)
if not messages:
    messages.append(HumanMessage(content=question))  # First message
response = llm.invoke(messages)  # Call LLM
messages.append(AIMessage(content=response.content))  # Store response
```
**Why?**
- Copy messages first (never modify input state directly)
- `HumanMessage` = user turn; `AIMessage` = LLM turn
- Building message history allows multi-turn flows

### 4. **Parsing LLM Output**
```python
if "sum" in response_text:
    operation = "sum"
    result = sum(numbers)
```
**Why?**
- LLM output is text, not structured data
- We extract/parse the intent (sum/multiply/divide)
- Then compute the result deterministically

## Summary

**What did we learn?**
1. How to load Vertex AI credentials from environment
2. How to initialize `ChatVertexAI` with deterministic temperature
3. How to use `HumanMessage` and `AIMessage` to build conversation
4. How to call LLM from a LangGraph node
5. How to parse LLM responses and extract structure

**Key differences from Lesson 5:**
- Lesson 5: Manual validation with if/else
- Lesson 6: LLM does the understanding and reasoning

## Why This Matters in Production

1. **Scalability**: LLM handles any arithmetic question, not just hardcoded patterns
2. **Natural Language**: Users can ask "What's 5+3?" or "Sum 5 and 3?" - both work
3. **Conversation History**: Message list enables complex multi-turn flows
4. **Deterministic Output**: temperature=0 ensures consistent arithmetic (doesn't say "maybe 8")
5. **Enterprise Security**: Vertex AI integrates with Google Cloud IAM and audit logging

**Next lesson:** We'll add **ToolNode** to make LLM call functions directly instead of parsing text.